# 3.1.9.2 Ручная отладка API -> RAW -> MON

Этот блокнот показывает ручной сценарий отладки:
1. fetch данных из API;
2. запись в `raw.vacancies` / `raw.quarantine`;
3. запись итогового run в `mon.pipeline_runs`;
4. SQL-проверки результата.

## 1) Setup окружения

In [6]:
import json
import sys
import uuid
from datetime import datetime, timezone

import psycopg2

# Путь к src проекта для импортов из notebook
project_src = "/Users/aleksshcherbakov/portfoilo/src"
if project_src not in sys.path:
    sys.path.append(project_src)

from trudvsem.fetch_layer import FetchConfig, SafetyLimits, TrudvsemFetcher
from trudvsem.raw_writer import RawBatchWriter, RawWriteContext
from trudvsem.run_metrics import (
    PipelineRunContext,
    PipelineRunsWriter,
    RunMetricsAccumulator,
)

## 2) Параметры БД и окна

In [21]:
# Для локального стенда можно использовать postgres/postgres
DB = {
    "host": "localhost",
    "port": 5432,
    "dbname": "project_db",
    "user": "postgres",
    "password": "postgres",
}

window_from = "2025-01-12T00:00:00+00:00"
window_to = "2025-12-31T23:59:59+00:00"
region_code = "66"  # Например: "77"
run_id = f"nb-debug-{uuid.uuid4()}"
run_id

'nb-debug-4ba54594-9ac8-4ffa-9c8f-5aba012e3ca8'

## 3) Fetch из API (1 страница)

In [22]:
fetcher = TrudvsemFetcher(
    FetchConfig(
        limit=20,
        safety=SafetyLimits(max_pages=1, max_items=100),
    )
)

pages = []
fetch_error = None
try:
    pages = list(
        fetcher.iter_pages(
            modified_from=window_from,
            modified_to=window_to,
            region_code=region_code,
        )
    )
except Exception as exc:
    fetch_error = str(exc)

print("pages:", len(pages))
print("requests_total:", fetcher.diagnostics.requests_total)
print("requests_success:", fetcher.diagnostics.requests_success)
print("http_5xx_count:", fetcher.diagnostics.http_5xx_count)
print("fetch_error:", fetch_error)

# Просмотр контента запроса/ответа для первой страницы
if pages:
    page0 = pages[0]
    print("request_url:", page0.request_url)
    print("request_params:", page0.request_params)
    print("http_status:", page0.http_status)
    print("response_meta:", page0.response_meta)
    print("raw_response_preview:")
    print(json.dumps(page0.raw_response, ensure_ascii=False)[:3000])

pages: 1
requests_total: 1
requests_success: 1
http_5xx_count: 0
fetch_error: None
request_url: http://opendata.trudvsem.ru/api/v1/vacancies/region/66?modifiedFrom=2025-01-12T00%3A00%3A00Z&modifiedTo=2025-12-31T23%3A59%3A59Z&offset=1&limit=20
request_params: {'modifiedFrom': '2025-01-12T00:00:00Z', 'modifiedTo': '2025-12-31T23:59:59Z', 'offset': 1, 'limit': 20}
http_status: 200
response_meta: {'total': 17817, 'limit': 20}
raw_response_preview:
{"status": "200", "request": {"api": "v1"}, "meta": {"total": 17817, "limit": 20}, "results": {"vacancies": [{"vacancy": {"id": "86fdf470-b014-11f0-b34f-9d0574b8f889", "source": "Вакансия интернет ресурса", "region": {"region_code": "6600000000000", "name": "Свердловская область"}, "company": {"companycode": "7226c750-02f1-11eb-8600-bfd13399602c", "hr-agency": false, "inn": "7718620740", "kpp": "997750001", "name": "МАГНИТ, Розничная сеть", "ogrn": "1067761906805", "url": "https://trudvsem.ru/company/7226c750-02f1-11eb-8600-bfd13399602c"}, "creat

## 4) Подготовка данных для записи

Если API недоступен или вернул 500, используем fixture-элемент, чтобы отладить запись в БД.
Контент запроса/ответа можно смотреть в `request_params`, `request_url`, `response_meta`, `raw_response`.


In [16]:
batches = []

if pages:
    for page in pages:
        batches.append(
            {
                "items": page.items,
                "request_params": page.request_params,
                "request_url": page.request_url,
                "http_status": page.http_status,
                "response_meta": page.response_meta,
                "raw_response": page.raw_response,
            }
        )
else:
    fixture_item = {
        "id": f"nb-vac-{uuid.uuid4()}",
        "date_modify": "2026-02-21T03:00:00+0300",
        "region": {"region_code": "7700000000000"},
        "name": "Notebook debug vacancy",
    }
    batches.append(
        {
            "items": [fixture_item],
            "request_params": {"offset": 1, "limit": 1},
            "request_url": None,
            "http_status": 200,
            "response_meta": {"note": "fixture-mode"},
            "raw_response": {
                "results": {"vacancies": [fixture_item]},
                "meta": {"note": "fixture-mode"},
            },
        }
    )

len(batches), len(batches[0]["items"])

(1, 20)

## 5) Запись в raw.* и сбор run-метрик

In [17]:
conn = psycopg2.connect(**DB)
raw_writer = RawBatchWriter(conn)
acc = RunMetricsAccumulator()
acc.add_fetch_diagnostics(fetcher.diagnostics)

write_results = []
for batch in batches:
    ctx = RawWriteContext(
        run_id=run_id,
        endpoint="/api/v1/vacancies",
        request_params=batch["request_params"],
        http_status=batch["http_status"],
        request_url=batch.get("request_url"),
        response_meta=batch.get("response_meta"),
    )
    result = raw_writer.write_batch(batch["items"], ctx)
    acc.add_batch_result(result)
    write_results.append(result)

write_results

[BatchWriteResult(items_total=20, raw_candidates=20, raw_rows_inserted=20, raw_rows_skipped_conflict=0, quarantine_candidates=0, quarantine_rows_inserted=0, max_source_modified_at=datetime.datetime(2026, 2, 21, 7, 4, 39, tzinfo=datetime.timezone.utc), max_ingested_at=datetime.datetime(2026, 2, 22, 6, 59, 43, 730714, tzinfo=datetime.timezone.utc))]

## 6) Запись в mon.pipeline_runs

In [18]:
run_ctx = PipelineRunContext(
    run_id=run_id,
    dag_id="raw_ingest_vacancies",
    run_type="manual",
    started_at=datetime.now(timezone.utc),
    window_from=datetime.fromisoformat(window_from),
    window_to=datetime.fromisoformat(window_to),
)

record = acc.build_record(
    context=run_ctx,
    has_critical_error=False,
    finished_at=datetime.now(timezone.utc),
)

PipelineRunsWriter(conn).write(record)
record

PipelineRunRecord(run_id='nb-debug-c77d6fc8-ebce-4fa3-9fa4-a9db83cadf68', dag_id='raw_ingest_vacancies', run_type='manual', status='partial', started_at=datetime.datetime(2026, 2, 22, 7, 0, 55, 613554, tzinfo=datetime.timezone.utc), finished_at=datetime.datetime(2026, 2, 22, 7, 0, 55, 613623, tzinfo=datetime.timezone.utc), duration_sec=0, window_from=datetime.datetime(2025, 12, 20, 0, 0, tzinfo=datetime.timezone.utc), window_to=datetime.datetime(2025, 12, 21, 23, 59, 59, tzinfo=datetime.timezone.utc), source_system='trudvsem', requests_total=1, requests_success=1, http_429_count=0, http_5xx_count=0, parse_error_count=0, items_extracted=20, raw_rows_inserted=20, raw_rows_skipped_conflict=0, quarantine_rows=0, max_source_modified_at=datetime.datetime(2026, 2, 21, 7, 4, 39, tzinfo=datetime.timezone.utc), max_ingested_at=datetime.datetime(2026, 2, 22, 6, 59, 43, 730714, tzinfo=datetime.timezone.utc), error_summary='safety stop reached: max_pages_reached')

## 7) SQL-проверки результата

In [19]:
with conn.cursor() as cur:
    cur.execute("select count(*) from raw.vacancies where run_id = %s", (run_id,))
    raw_count = cur.fetchone()[0]

    cur.execute("select count(*) from raw.quarantine where run_id = %s", (run_id,))
    quarantine_count = cur.fetchone()[0]

    cur.execute("""
        select status, requests_total, requests_success, items_extracted,
               raw_rows_inserted, raw_rows_skipped_conflict, quarantine_rows
        from mon.pipeline_runs
        where run_id = %s
    """, (run_id,))
    run_row = cur.fetchone()

raw_count, quarantine_count, run_row

(20, 0, ('partial', 1, 1, 20, 20, 0, 0))

## 8) Завершение

In [20]:
conn.close()
"done"

'done'